# Bradford Bulls — Team-Aware Frame Extraction v3.0

### Pipeline
```
Phase 0A: Detect static overlays (scoreboard, watermark)
Phase 0B: Semi-supervised team calibration (YOU label a few samples)
Pass 1:  Fast scan → find quality segments
Pass 2:  Team-aware extraction + quota selection
Save:    Frames + enriched metadata CSV
```

**New in v3**: Instead of unreliable auto-clustering, you now label 3-5 jersey
samples from a numbered grid. The system learns from YOUR examples.

## 0. Install Dependencies
Run this cell, then **Restart Runtime**.

In [ ]:
!pip install -q ultralytics>=8.3.0 scikit-learn imagehash scikit-image pillow opencv-python-headless roboflow
print("\nDone. Please RESTART RUNTIME now (Runtime > Restart runtime).")

## 1. YOUR CONFIG

In [ ]:
MEMBER_NAME = "your_name"
VIDEO_FILENAME = "M01_white_1080p.mp4"
TEST_MODE = True

## 2. Setup Environment

In [ ]:
import torch, cv2, os, sys, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected.")

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/Bradford_Bulls")
VIDEOS_DIR = DRIVE_ROOT / "videos"
FRAMES_DIR = DRIVE_ROOT / "frames"
METADATA_DIR = DRIVE_ROOT / "metadata"
REPORTS_DIR = DRIVE_ROOT / "reports"
SRC_DIR = DRIVE_ROOT / "src"

for d in [VIDEOS_DIR, FRAMES_DIR, METADATA_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Add src to Python path
if (SRC_DIR / "frame_extraction").exists():
    sys.path.insert(0, str(SRC_DIR))
    print(f"Using modules from Drive: {SRC_DIR}")
elif Path("/content/BRADFORD_BULLS_PROJECT/src").exists():
    sys.path.insert(0, "/content/BRADFORD_BULLS_PROJECT/src")
    print("Using modules from cloned repo")
else:
    raise FileNotFoundError("src/frame_extraction not found. Upload src/ to Drive or clone repo.")

from frame_extraction.overlay import detect_static_overlays, visualize_overlay
from frame_extraction.calibration import collect_samples, show_samples, build_calibration
from frame_extraction.pipeline import pass1_fast_scan, pass2_extract
from frame_extraction.selection import select_by_quota, print_selection_summary

MATCH_ID = Path(VIDEO_FILENAME).stem
VIDEO_PATH = VIDEOS_DIR / VIDEO_FILENAME
FRAMES_OUTPUT_DIR = FRAMES_DIR / MATCH_ID
FRAMES_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH}"
print(f"\nVideo: {VIDEO_PATH.name} ({VIDEO_PATH.stat().st_size/1e6:.0f} MB)")

## 3. Load Video & Model

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

PERSON_DETECTION_MODEL = "yolo11l.pt"
JPEG_QUALITY = 95

if TEST_MODE:
    MAX_SCAN_FRAMES = 2000
    TARGET_FRAMES = 40
else:
    MAX_SCAN_FRAMES = TOTAL_FRAMES
    TARGET_FRAMES = 350

print(f"Duration: {TOTAL_FRAMES/FPS/60:.1f}min | {W}x{H} | {FPS:.0f}fps | {TOTAL_FRAMES:,} frames")
print(f"Mode: {'TEST' if TEST_MODE else 'PRODUCTION'} | Scan: {MAX_SCAN_FRAMES:,} | Target: {TARGET_FRAMES}")

print(f"\nLoading {PERSON_DETECTION_MODEL}...")
yolo_model = YOLO(PERSON_DETECTION_MODEL)
print("Model loaded.")

## 4. Detect Static Overlays
Finds scoreboard/watermark by analyzing which pixels stay constant.

In [ ]:
overlay_mask, overlay_ratio = detect_static_overlays(VIDEO_PATH)
print(f"Overlay coverage: {overlay_ratio*100:.1f}% masked")

cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, TOTAL_FRAMES // 3)
ret, sample_frame = cap.read()
cap.release()
if ret:
    visualize_overlay(sample_frame, overlay_mask)

## 5A. Collect & Display Torso Samples

This samples diverse player torso crops from the video.

**Look at the numbered grid** and write down which numbers show **YOUR team's jersey**.

In [ ]:
sample_data = collect_samples(
    VIDEO_PATH, yolo_model, DEVICE,
    overlay_mask=overlay_mask,
    n_sample_frames=80,
    n_display=24,
)

show_samples(sample_data)

## 5B. Label YOUR Team

Look at the numbered grid above. **Edit `MY_TEAM` below** with the numbers
that show YOUR team's jersey (Bradford Bulls = dark/black striped jersey).

You need **at least 3** numbers. More is better (5-8 ideal).

Example: `MY_TEAM = [0, 3, 5, 8, 11, 14]`

In [ ]:
# ✏️ EDIT THIS LIST — Put the grid numbers that show YOUR team
MY_TEAM = [0, 1, 2]  # ← Change these numbers!

calibration = build_calibration(sample_data, MY_TEAM)

# The verification grid below shows the result.
# Top row = your team, Bottom row = opponent.
# Gold borders = samples you labeled.
# If it looks wrong, go back and fix MY_TEAM.

## 6. Pass 1 — Fast Scan

In [ ]:
segments, zoom_timeline, video_info = pass1_fast_scan(
    VIDEO_PATH, yolo_model, DEVICE, max_frames=MAX_SCAN_FRAMES
)

print(f"\nScanned: {len(zoom_timeline):,} frames")
print(f"Quality segments: {len(segments)}")
if segments:
    avg_len = np.mean([(s[-1][0]-s[0][0])/FPS for s in segments])
    print(f"Avg segment length: {avg_len:.1f}s")

## 7. Pass 2 — Team-Aware Extraction

In [ ]:
candidates, pass2_stats = pass2_extract(
    VIDEO_PATH, segments, yolo_model, DEVICE,
    calibration, overlay_mask, video_info,
    max_frames=MAX_SCAN_FRAMES
)

print(f"\nCandidates: {len(candidates)}")
print(f"  Target: {pass2_stats['team_target']} | Mixed: {pass2_stats['team_mixed']} | Opponent: {pass2_stats['team_opponent']}")
print(f"  Skipped: pitch={pass2_stats['skipped_pitch']}, blur={pass2_stats['skipped_blurry']}")

## 8. Quota Selection

In [ ]:
selected, sel_stats = select_by_quota(candidates, TARGET_FRAMES)
print_selection_summary(selected, sel_stats, len(candidates))

## 9. Save Frames & Metadata

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
rows = []

for idx, meta in enumerate(tqdm(selected, desc="Saving")):
    cap.set(cv2.CAP_PROP_POS_FRAMES, meta["frame_num"])
    ret, frame = cap.read()
    if not ret: continue

    fname = f"{MATCH_ID}_{meta['frame_num']:06d}_{meta['timestamp_hms']}.jpg"
    cv2.imwrite(str(FRAMES_OUTPUT_DIR / fname), frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])

    rows.append({"filename": fname, **meta, "match_id": MATCH_ID,
                 "member": MEMBER_NAME, "extracted_at": datetime.now().isoformat()})

cap.release()

df = pd.DataFrame(rows)
csv_path = METADATA_DIR / f"{MATCH_ID}_v3_index.csv"
df.to_csv(csv_path, index=False)
print(f"\nSaved {len(df)} frames to: {FRAMES_OUTPUT_DIR}\nMetadata: {csv_path}")

## 10. Preview by Category

In [ ]:
for cat in sorted(df["category"].unique()):
    cat_df = df[df["category"] == cat].sort_values("score", ascending=False)
    n_show = min(4, len(cat_df))
    if n_show == 0: continue

    fig, axes = plt.subplots(1, n_show, figsize=(5*n_show, 5))
    if n_show == 1: axes = [axes]

    for i, (_, row) in enumerate(cat_df.head(n_show).iterrows()):
        fpath = FRAMES_OUTPUT_DIR / row["filename"]
        if fpath.exists():
            img = cv2.imread(str(fpath))
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            axes[i].set_title(f"{row['timestamp_hms']} | {row['score']:.2f}\ntarget={row['n_target']} opp={row['n_opponent']}", fontsize=8)
        axes[i].axis("off")

    plt.suptitle(f"{cat.upper()} ({len(cat_df)} frames)", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

print(f"\nCategory distribution:\n{df['category'].value_counts().to_string()}")
print(f"\nShot type distribution:\n{df['shot_type'].value_counts().to_string()}")

## 11. Quality Charts

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0,0].hist(df["sharpness"], bins=25, color="steelblue", edgecolor="white")
axes[0,0].set_title("Sharpness")
axes[0,1].hist(df["score"], bins=25, color="forestgreen", edgecolor="white")
axes[0,1].set_title("Quality Score")
axes[0,2].hist(df["team_dominance"], bins=20, color="coral", edgecolor="white")
axes[0,2].set_title("Team Dominance")
axes[1,0].hist(df["max_person_area_ratio"], bins=25, color="mediumpurple", edgecolor="white")
axes[1,0].set_title("Largest Person Size")
axes[1,1].hist(df["timestamp_sec"]/60, bins=25, color="goldenrod", edgecolor="white")
axes[1,1].set_title("Temporal Distribution (min)")
cat_counts = df["category"].value_counts()
axes[1,2].barh(cat_counts.index, cat_counts.values, color="teal")
axes[1,2].set_title("Frames per Category")
plt.suptitle(f"{MATCH_ID} — {len(df)} frames", fontsize=14)
plt.tight_layout()
plt.show()

## 12. Upload to Roboflow (Optional)

In [ ]:
import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = getpass.getpass("Roboflow API Key: ")
ROBOFLOW_PROJECT = "kit-sponsor-logos"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace().project(ROBOFLOW_PROJECT)

for fp in tqdm(sorted(FRAMES_OUTPUT_DIR.glob(f"{MATCH_ID}_*.jpg")), desc="Uploading"):
    try:
        project.upload(image_path=str(fp), split="train", tag_names=[f"{MATCH_ID},{MEMBER_NAME},v3"])
    except Exception as e:
        print(f"Error: {e}")
        break

print("Upload complete.")